# 1. Import the necessary libraries

In [ ]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from mesh.core import *
from mesh.MESH_old import *
from problems.benchmark_problems import get_problem
from runners import get_tuned_parameters

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PolynomialMutation
from pymoo.optimize import minimize

import numpy as np
import pandas as pd
import statistics
import timeit

# 2. Experiment Configuration

In [ ]:
#################### CUSTOMIZABLE ####################

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [
    ('zdt1', 2, 10, None), ('zdt2', 2, 10, None), ('zdt3', 2, 10, None), ('zdt4', 2, 10, None), ('zdt6', 2, 10, None),
    ('dtlz1', 3, 10, None), ('dtlz2', 3, 10, None), ('dtlz3', 3, 10, None), ('dtlz4', 3, 10, None), ('dtlz5', 3, 10, None), ('dtlz6', 3, 10, None), ('dtlz7', 3, 10, None),
    ('wfg1', 3, 10, 6), ('wfg2', 3, 10, 6), ('wfg3', 3, 10, 6), ('wfg4', 3, 10, 6), ('wfg5', 3, 10, 6), ('wfg6', 3, 10, 6), ('wfg7', 3, 10, 6), ('wfg8', 3, 10, 6),
    ('wfg9', 3, 10, 6)]

# Execution configuration
num_repeats = 30 # Number of runs for collecting statistics
max_fitness_eval = 15000 # Maximum fitness evaluations (not used if it is None)
population_size = 100 # Population size
random_state = None # Set a seed for random numbers (not used if it is None)

tuning_folder = f'../hyperparams' # Folder to get the tuned parameters
######################################################

df_columns = ['Function', 'Algorithm', 'Min Time (s)', 'Max Time (s)', 'Mean Time (s)', 'Std Dev Time (s)', 'Median Time (s)']

# 3. Auxiliar Functions

In [ ]:
# Function to get the fitness function configuration
def get_exp_problem(func_name, objective_dim, position_dim, wfg_k):
	func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim, wfg_k=wfg_k)
	return func, objective_dim, position_dim, position_min_value, position_max_value

def run_mesh_vectorized(experiment: tuple, # Information to run the experiments
											# (experiment name, experiment folder, fine tuning folder, number of runs, maximum fitness evaluations, population size, random seed)
						problem: tuple, # Problem setup (fitness function, number of objectives, number of decision variables, lower bound array, upper bound array)
						fixed_parameters: tuple, # MESH fixed parameters
						tunable_parameters: tuple # MESH tunable parameters
						) -> None:
	# Get the experiment name and folder to store results
	experiment_configuration, fine_tuning_folder, max_fitness_eval, population_size, random_state = experiment

  	# Get the problem
	fit_function, objective_dim, position_dim, lower_bound_array, upper_bound_array = problem

	# Get the fixed parameters
	memory_size, global_best_attribution_type, dm_pool_type, dm_operation_type = fixed_parameters

	# Get tunable parameters (check if the parameters was tuned)
	tuned_parameters_dict = get_tuned_parameters(experiment_configuration, fine_tuning_folder)
	communication_probability = tuned_parameters_dict['communication_probability'] if ('communication_probability' in tuned_parameters_dict) else tunable_parameters[0]
	mutation_rate = tuned_parameters_dict['mutation_rate'] if ('mutation_rate' in tuned_parameters_dict) else tunable_parameters[1]
	personal_guide_array_size = tuned_parameters_dict['personal_guide_array_size'] if ('personal_guide_array_size' in tuned_parameters_dict) else tunable_parameters[2]

	params = MeshParameters(objective_dim = objective_dim,
							position_dim = position_dim,
							position_lower_bounds = lower_bound_array,
							position_upper_bounds = upper_bound_array, 
							population_size = population_size,
							memory_size = memory_size,
							global_guide_method = global_best_attribution_type,
							dm_pool_type = dm_pool_type,
							dm_operation_type = dm_operation_type,
							communication_probability = communication_probability,
							mutation_rate = mutation_rate,
							max_gen = None,
							max_fit_eval = max_fitness_eval,
							max_personal_guides = personal_guide_array_size,
							random_state = random_state)
	mesh = Mesh(params = params, fitness_function = fit_function)
	mesh.run()

def run_mesh_old(experiment: tuple, # Information to run the experiments
									# (experiment name, experiment folder, fine tuning folder, number of runs, maximum fitness evaluations, population size, random seed)
				problem: tuple, # Problem setup (fitness function, number of objectives, number of decision variables, lower bound array, upper bound array)
				fixed_parameters: tuple, # MESH fixed parameters
				tunable_parameters: tuple # MESH tunable parameters
				) -> None:

	# Get the experiment name and folder to store results
	experiment_configuration, fine_tuning_folder, max_fitness_eval, population_size, random_state = experiment

  	# Get the problem
	fit_function, objective_dim, position_dim, lower_bound_array, upper_bound_array = problem

	# Get the fixed parameters
	memory_size, global_best_attribution_type, dm_pool_type, dm_operation_type = fixed_parameters

	# Get tunable parameters (check if the parameters was tuned)
	tuned_parameters_dict = get_tuned_parameters(experiment_configuration, fine_tuning_folder)
	communication_probability = tuned_parameters_dict['communication_probability'] if ('communication_probability' in tuned_parameters_dict) else tunable_parameters[0]
	mutation_rate = tuned_parameters_dict['mutation_rate'] if ('mutation_rate' in tuned_parameters_dict) else tunable_parameters[1]
	personal_guide_array_size = tuned_parameters_dict['personal_guide_array_size'] if ('personal_guide_array_size' in tuned_parameters_dict) else tunable_parameters[2]

	params_old = MESH_Params_old(objectives_dim = objective_dim,
								optimizations_type = [False]*objective_dim,
								max_iterations = 0,
								max_fitness_eval = max_fitness_eval,
								position_dim = position_dim,
								position_max_value = upper_bound_array,
								position_min_value = lower_bound_array,
								population_size = population_size,
								memory_size = memory_size,
								memory_update_type = 0,
								global_best_attribution_type = global_best_attribution_type,
								DE_mutation_type = dm_operation_type,
								Xr_pool_type = dm_pool_type,
								crowd_distance_type = 0,
								communication_probability = communication_probability,
								mutation_rate = mutation_rate,
								personal_guide_array_size = personal_guide_array_size,
								random_state=random_state)
	old_mesh = MESH_old(params_old, fit_function)
	old_mesh.log_memory = None
	old_mesh.run()

def run_nsga2(experiment: tuple, # Information to run the experiments
								 # (experiment name, experiment folder, fine tuning folder, number of runs, maximum fitness evaluations, population size, random seed)
			  problem: tuple, # Problem setup (fitness function, number of objectives, number of decision variables, lower bound array, upper bound array)
			  fixed_parameters: tuple, # MESH fixed parameters
			  tunable_parameters: tuple # MESH tunable parameters
			) -> None:

	# Get the experiment name and folder to store results
	experiment_configuration, fine_tuning_folder, max_fitness_eval, population_size, random_state = experiment

  	# Get the problem
	fit_function, objective_dim, position_dim, lower_bound_array, upper_bound_array = problem
	class MyProblem(Problem):
		def __init__(self, n_var, n_obj, xl, xu):
			super().__init__(n_var=n_var, n_obj=n_obj, n_constr=0, xl=xl, xu=xu)
		def _evaluate(self, X, out, *args, **kwargs):
			out["F"] = np.array([fit_function(x) for x in X])

	nsga2_fit_function = MyProblem(n_obj=objective_dim, n_var=position_dim, xl=lower_bound_array, xu=upper_bound_array)

	# Get tunable parameters (check if the parameters was tuned)
	tuned_parameters_dict = get_tuned_parameters(experiment_configuration, fine_tuning_folder)
	recombination_probability = tuned_parameters_dict['recombination_probability'] if ('recombination_probability' in tuned_parameters_dict) else tunable_parameters[0]
	eta_recombination = tuned_parameters_dict['eta_recombination'] if ('eta_recombination' in tuned_parameters_dict) else tunable_parameters[1]
	mutation_probability = tuned_parameters_dict['mutation_probability'] if ('mutation_probability' in tuned_parameters_dict) else tunable_parameters[2]
	eta_mutation = tuned_parameters_dict['eta_mutation'] if ('eta_mutation' in tuned_parameters_dict) else tunable_parameters[3]

	# Instantiate NSGA2
	crossover = SBX(prob=recombination_probability, prob_var=1.0, eta=eta_recombination)
	mutation = PolynomialMutation(prob=mutation_probability, eta=eta_mutation)
	nsga2 = NSGA2(pop_size=population_size,
								crossover=crossover,
								mutation=mutation,
								eliminate_duplicates=True)

	# Execute NSGA2
	minimize(nsga2_fit_function, nsga2, ('n_eval', max_fitness_eval), seed=random_state, verbose=False)

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		run_function(*params)
	except Exception as e:
		print(f'Error: {str(e)}')

# 4. Benchmark

## 4.1 MESH

In [ ]:
#################### CUSTOMIZABLE ####################
# MESH fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory
config_list = [(0,1,2), (1,1,1), (1,0,0), (0,0,0), (0,2,1),
               (1,1,0), (0,1,0), (0,0,1), (1,2,0), (1,2,3), (0,1,0), (1,0,0),
               (1,2,0), (1,0,0), (0,2,0), (1,2,2), (1,1,3), (1,1,0), (0,2,2), (1,0,2), (0,0,0)]


# MESH tunable parameters
# OBS: The function "run_mesh" and "run_mesh_old" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
mesh_communication_probability = 0.8 # Communication probability
mesh_mutation_rate = 0.9 # Mutation rate
mesh_personal_guide_array_size = 1 # Number of personal guides
######################################################

info_list = [(exp[0].upper(), f'G{config_list[i][0]+1}S{config_list[i][1]+1}D{config_list[i][2]+1}')
			 for i, exp in enumerate(experiments)
]

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'MESH_G{config_list[i][0]+1}S{config_list[i][1]+1}D{config_list[i][2]+1}_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
    get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
    (mesh_memory_size, *config_list[i]),
    (mesh_communication_probability, mesh_mutation_rate, mesh_personal_guide_array_size)]
    for i, exp in enumerate(experiments)
]

# Collect the times for each experiment
mesh_times = [
    timeit.repeat(
        lambda: safe_run(run_mesh_vectorized, params),
        repeat=num_repeats,
        number=1
    )
    for params in params_list
]

# Create a DataFrame to store the results
mesh_results = [
  [info_list[i][0],
   'MESH ' + info_list[i][1],
   min(t),
   max(t),
   statistics.mean(t),
   statistics.stdev(t) if len(t) > 1 else 0.0,
   statistics.median(t)] for i, t in enumerate(mesh_times)]

mesh_df = pd.DataFrame(mesh_results, columns=df_columns)
mesh_df

  0%|          | 0/15000 [00:00<?, ?it/s]                

In [ ]:
print(mesh_df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))

### Previous MESH

In [ ]:
# MESH OLD
config_list = [(0,1,1), (0,0,0), (1,1,1), (0,0,1), (0,2,1),
               (0,2,4), (1,1,4), (0,1,4), (1,1,0), (0,0,0), (1,0,1), (0,2,1),
               (0,2,4), (1,0,3), (0,1,3), (0,0,2), (1,2,3), (0,0,2), (0,0,2), (0,2,3), (1,0,3)]

info_list = [(exp[0].upper(), f'G{config_list[i][0]+1}S{config_list[i][1]+1}D{config_list[i][2]+1}')
			 for i, exp in enumerate(experiments)
]

In [ ]:
# MESH OLD
num_repeats = 5

# Set the list of parameters
params_list = [
  	[(F'OLD_MESH_G{config_list[i][0]+1}S{config_list[i][1]+1}D{config_list[i][2]+1}_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
    get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
    (mesh_memory_size, *config_list[i]),
    (mesh_communication_probability, mesh_mutation_rate, mesh_personal_guide_array_size)]
    for i, exp in enumerate(experiments)
]

# Collect the times for each experiment
mesh_old_times = [
    timeit.repeat(
        lambda: safe_run(run_mesh_old, params),
        repeat=num_repeats,
        number=1
    )
    for params in params_list
]

# Create a DataFrame to store the results
mesh_old_results = [
  [info_list[i][0],
   'MESH ' + info_list[i][1],
   min(t),
   max(t),
   statistics.mean(t),
   statistics.stdev(t) if len(t) > 1 else 0.0,
   statistics.median(t)] for i, t in enumerate(mesh_old_times)]

mesh_old_df = pd.DataFrame(mesh_old_results, columns=df_columns)
mesh_old_df

In [ ]:
print(mesh_old_df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))

# 4.2 NSGA-II

In [ ]:
#################### CUSTOMIZABLE ####################

# NSGA2 tunable parameters
# OBS: The function "run_nsga2" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
nsga2_recombination_probability = 0.45
nsga2_eta_recombination = 19
nsga2_mutation_probability = 0.06
nsga2_eta_mutation = 14.7
######################################################

# Set the list of parameters
params_list = [
  	[(F'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
	 get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 (),
	 (nsga2_recombination_probability, nsga2_eta_recombination, nsga2_mutation_probability, nsga2_eta_mutation)]
     for exp in experiments
]

# Collect the times for each experiment
nsga2_times = [
    timeit.repeat(
        lambda: safe_run(run_nsga2, params),
        repeat=num_repeats,
        number=1
    )
    for params in params_list
]

# Create a DataFrame to store the results
nsga2_results = [
  [experiments[i][0].upper(),
   'NSGA2',
   min(t),
   max(t),
   statistics.mean(t),
   statistics.stdev(t) if len(t) > 1 else 0.0,
   statistics.median(t)] for i, t in enumerate(nsga2_times)]

nsga2_df = pd.DataFrame(nsga2_results, columns=df_columns)
nsga2_df

In [ ]:
print(nsga2_df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))